<!-- 학습 보강 셀 -->

# 09. Persisting VectorStoreIndex 학습 흐름

이 노트북은 만든 인덱스를 디스크에 저장하고 다시 불러오는 과정을 다룹니다.
실제 RAG 서비스에서는 매번 문서를 다시 임베딩하면 비용과 시간이 크기 때문에 저장/로드가 필수입니다.

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [ ]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model='gemma2:2b',
    temperature=0,
    request_timeout=120,
)

embed_model = OllamaEmbedding(
    model_name='nomic-embed-text',
)

In [ ]:
# 데이터 로드
# - 저장/로드 예제에서는 pdf_sample2의 논문을 사용합니다.
documents = SimpleDirectoryReader('../NewData/pdf_sample2/').load_data()
print('읽어온 문서 수:', len(documents))

In [ ]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

<!-- 학습 보강 셀 -->

## 저장하기 전과 저장한 후의 차이

메모리에 있는 `index`는 현재 커널이 살아 있는 동안만 사용할 수 있습니다.
`persist()`로 저장하면 커널을 재시작해도 같은 인덱스를 다시 로드할 수 있습니다.

In [ ]:
# 인덱스 저장
# - storage_context.persist()는 docstore, index_store, vector_store 정보를 디렉토리에 저장합니다.
# - 저장 후에는 문서를 다시 임베딩하지 않고 load_index_from_storage로 불러올 수 있습니다.
persist_dir = './saved_index'
index.storage_context.persist(persist_dir=persist_dir)

<!-- 학습 보강 셀 -->

## 저장 폴더 안의 파일들이 의미하는 것

저장 폴더에는 문서 조각 정보, 인덱스 구조, 벡터 스토어 데이터가 나뉘어 들어갑니다.
파일 하나만 복사해서는 로드가 실패할 수 있으므로, 저장 디렉토리 전체를 하나의 인덱스 묶음으로 다루는 것이 안전합니다.

In [ ]:
# 저장된 인덱스 로드 위치 정의
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)

<!-- 학습 보강 셀 -->

## 로드할 때 같은 임베딩 모델을 써야 하는 이유

저장된 벡터는 특정 임베딩 모델이 만든 좌표계 위에 있습니다.
다른 임베딩 모델로 질문 벡터를 만들면 좌표계가 달라져 검색 품질이 크게 떨어질 수 있습니다.
따라서 생성과 로드 시 같은 임베딩 모델을 사용하는 것이 기본 원칙입니다.

In [ ]:
# 인덱스 로드
# - 생성할 때 사용한 임베딩 모델과 같은 모델을 지정해야 검색 결과가 일관됩니다.
loaded_index = load_index_from_storage(
    storage_context,
    embed_model=embed_model,
)

In [ ]:
# 로드한 인덱스로 쿼리 엔진 생성
loaded_query_engine = loaded_index.as_query_engine(llm=llm)

In [ ]:
# 저장소에서 불러온 인덱스로 질의 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = loaded_query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)

#### 메모리 인덱스와 저장소 로드 인덱스 비교

아래 셀은 방금 메모리에 만든 `index`로 같은 질문을 실행해 저장소에서 불러온 결과와 비교하는 용도입니다.

In [ ]:
# 메모리에 있는 원본 인덱스로 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

In [ ]:
# 메모리 인덱스로 같은 질문 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)